# Rule-Based Generation

**Domain:** Procedural Generation  ·  **from study list**  ·  **runnable:** yes

A refresher on **rule-based generation**: build or modify content by repeatedly applying a set
of explicit **condition → action** rules (a *production system*) until nothing more fires. The
umbrella paradigm that grammars, L-systems, and cellular automata are all special cases of — you
author the *rules*, an *engine* matches and applies them, and structured content falls out. The
opposite of learned/statistical generation: every output is traceable to the rules that produced
it.

## 1. What & Why

**Rule-based generation** creates content by running a **production system**: a working set of
facts (a grid, a string, a scene graph, a bag of attributes) plus a list of **rules**, each a
`IF <condition> THEN <action>` pair. An **engine** runs the *recognize–act cycle*: find every
rule whose condition matches the current state, pick one (by priority / order / random), apply
its action to mutate the state, and repeat until no rule matches (a **fixpoint**) or a budget
runs out.

**The problem it solves.** You want generated content that is *authorable, explainable, and
controllable* — a designer can read the rules, predict roughly what they do, and tweak one rule
to change one behavior. No training data, no GPU, no black box. "If a room has no exit, add a
door." "If depth is a multiple of 5, spawn a boss." "If a cell is floor with one neighbor, it's a
dead-end — put treasure there." Each rule is a small, testable unit of design intent.

**When to reach for it.** Decorating/validating maps (place doors, torches, loot by local
conditions), encounter/loot/NPC tables with conditional logic, quest and dialogue assembly,
mission grammars, "juice" passes that post-process another generator's raw output, and any
pipeline where designers must *edit the behavior directly* rather than retrain a model.

**When not to.** When you have *examples* but can't articulate rules — learn statistics with a
Markov chain ([[markov-chains]]) or a neural model. When the structure is recursive *text/syntax*,
a context-free grammar ([[context-free-grammars]]) or L-system ([[l-systems]]) is the right
specialization. When neighbor constraints over a 2D/3D tile set must be solved globally, use Wave
Function Collapse ([[wave-function-collapse]]). And beware: large rule sets develop emergent,
order-dependent interactions that get hard to reason about — the classic expert-system trap.

## 2. Mental Model

Think of a **spreadsheet full of "if this, then that" sticky notes, plus a robot that keeps
walking the board**. The board is your *working memory* (the current map / facts). Each sticky
note is a rule. The robot scans for any note whose IF is currently true, does its THEN (changing
the board), and starts over. When a full pass finds no applicable note, it stops — that stable
state is your output.

```
            ┌─────────────────────────────┐
            │   Working memory (state)    │ ◀── actions mutate it
            │   grid / facts / string     │
            └─────────────┬───────────────┘
                          │ match
                 ┌────────▼─────────┐
                 │  Rule set        │   recognize → resolve → act
   IF cond ──▶   │  IF c1 THEN a1   │   (repeat until fixpoint
   THEN act      │  IF c2 THEN a2   │    or step budget hit)
                 │  IF c3 THEN a3   │
                 └──────────────────┘
```

The three knobs that define an engine: **matching** (which rules are eligible), **conflict
resolution** (when several match, which one fires — priority/salience, rule order, specificity,
or random), and **termination** (fixpoint, a step cap, or a goal test). Grammars, L-systems, and
cellular automata are just this loop with the conditions/actions specialized: a grammar matches a
non-terminal and replaces it; an L-system applies *all* rules in parallel each step; a CA's rule
is "look at my neighbors, decide my next state.\"

## 3. Key Concepts

- **Production rule** — `IF condition THEN action`. The condition is a predicate over the current
  state; the action mutates it (add/replace/remove). Also called a *production* or *if-then rule*.
- **Working memory** — the mutable state the rules read and write: a grid, a string, a list of
  facts, a scene graph. Generation = transforming working memory until stable.
- **Recognize–act cycle** — the engine loop: (1) *recognize* all rules whose conditions hold,
  (2) *resolve* the conflict set down to one (or apply all), (3) *act*, then repeat.
- **Conflict resolution** — the policy for choosing among simultaneously-matching rules:
  **salience/priority** (highest first), **rule order** (top-down), **specificity** (most specific
  condition wins), or **random/weighted** for variety. This is where most behavior actually lives.
- **Forward chaining** — data-driven: start from facts, fire rules, derive new facts. (Most PCG.)
  **Backward chaining** is goal-driven: start from a desired output, find rules that produce it.
- **Salience** — an integer priority on a rule; higher salience fires first. The standard way to
  layer "important structural rules before cosmetic ones."
- **Fixpoint / quiescence** — the state where no rule's condition matches anymore. Reaching it
  cleanly (and provably) is the central design problem; otherwise cap the number of steps.
- **Parallel vs sequential application** — apply one rule per step (sequential, order matters) or
  every matching rule at once against a snapshot (parallel, like CA / L-systems). Different model,
  different results.
- **Constructive vs constraint rules** — *constructive* rules add content ("place a door");
  *constraint* rules reject or repair states that violate invariants ("no two bosses adjacent").

## 4. Setup

Nothing to install — a production system is a list of `(condition, action)` callables plus a loop,
all standard-library Python. We use `random` for weighted/variety rules and `dataclasses` to give
rules a readable shape.

For production engines you'd reach for a real rules system —
[`experta`](https://pypi.org/project/experta/) (a Python CLIPS-style forward-chaining engine) or
[`durable_rules`](https://pypi.org/project/durable-rules/) — but the core loop is a dozen lines,
so we build it here to keep the idea transparent and dependency-free.

In [1]:
# Rule-based generation needs only the standard library. Real engines (optional):
#   %pip install -q experta        # CLIPS-style forward-chaining production system
import os
import random
from dataclasses import dataclass, field
from typing import Callable

rng = random.Random(7)  # seeded so output is reproducible
print("stdlib only — ready")

stdlib only — ready


## 5. Worked Examples

Two self-contained examples that show the two faces of rule-based generation:

1. **A dungeon-decoration engine** — *constructive spatial rules* run to a **fixpoint** over a
   grid. Local `IF neighbor-pattern THEN place-feature` rules turn a bare map into a decorated
   one. This is the recognize–act cycle on 2D working memory.
2. **A priority-driven encounter generator** — *salience + guard conditions* with real
   **conflict resolution** and **forward chaining**, where one rule's action enables another's
   condition. Same engine, abstract facts instead of a grid.

### Example 1 — Dungeon decoration as a production system

The working memory is a small dungeon (`#` wall, `.` floor). Each rule looks at a floor cell's
orthogonal neighborhood and, if a pattern matches, stamps a feature. The engine re-scans until a
full pass changes nothing — a clean fixpoint, because a decorated cell no longer matches the
"plain floor" guard. This is exactly how a CA or grammar works, just with hand-authored local
rules.

In [2]:
raw = [
    "###########",
    "#....#....#",
    "#.##.#.##.#",
    "#.#......##",
    "#.#.###.#.#",
    "#...#.#...#",
    "###.#.#.###",
    "#...#.#...#",
    "###########",
]
grid = [list(row) for row in raw]
H, W = len(grid), len(grid[0])

OPEN = set(".T+o")          # cells that count as walkable when matching neighbors

def neighbors_open(r, c):
    """Count orthogonal neighbors that are walkable."""
    n = 0
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        rr, cc = r + dr, c + dc
        if 0 <= rr < H and 0 <= cc < W and grid[rr][cc] in OPEN:
            n += 1
    return n

@dataclass
class CellRule:
    name: str
    when: Callable[[int, int], bool]   # condition on (row, col)
    then: str                          # feature char to stamp

# IF plain floor with exactly one open neighbor (a dead-end) THEN treasure.
# IF plain floor with >= 3 open neighbors (a junction)        THEN torch.
rules = [
    CellRule("dead_end_treasure", lambda r, c: grid[r][c] == "." and neighbors_open(r, c) == 1, "T"),
    CellRule("junction_torch",    lambda r, c: grid[r][c] == "." and neighbors_open(r, c) >= 3, "+"),
]

def run_to_fixpoint(rules, max_steps=200):
    fired = 0
    for _ in range(max_steps):
        changed = False
        for r in range(H):
            for c in range(W):
                for rule in rules:                 # rule order = conflict resolution
                    if rule.when(r, c):
                        grid[r][c] = rule.then      # act: mutate working memory
                        fired += 1
                        changed = True
                        break                       # one rule per cell per pass
        if not changed:                             # no rule matched anywhere -> fixpoint
            break
    return fired

print("before:")
print("\n".join("".join(row) for row in (list(r) for r in raw)))
fired = run_to_fixpoint(rules)
print(f"\nafter {fired} rule firings  (T = treasure in a dead-end, + = torch at a junction):")
print("\n".join("".join(row) for row in grid))

before:
###########
#....#....#
#.##.#.##.#
#.#......##
#.#.###.#.#
#...#.#...#
###.#.#.###
#...#.#...#
###########

after 12 rule firings  (T = treasure in a dead-end, + = torch at a junction):
###########
#....#....#
#.##.#.##T#
#.#.+.++T##
#.#.###.#T#
#..+#T#+..#
###.#.#.###
#T..#T#..T#
###########


Two local rules, zero global planning, and the map is decorated consistently everywhere the
pattern occurs. Add a `door` rule (`wall between two floors`) or a `trap` rule and the behavior
composes — that incremental authorability is the whole point. Because every feature char leaves
the `== "."` guard, the system provably reaches a fixpoint in one effective pass.

### Example 2 — Priority rules with forward chaining

Now the working memory is a bag of *facts* about a dungeon region. Rules carry a **salience**
(priority); the engine fires the highest-salience eligible rule, mutates the facts, and re-scans
— so a rule's action can enable a later rule's condition (**forward chaining**). This is the
pattern behind encounter tables, loot rolls, and quest assembly.

In [3]:
@dataclass
class Rule:
    name: str
    salience: int
    when: Callable[[dict], bool]    # condition over the facts
    then: Callable[[dict], None]    # action mutates the facts

rules = [
    # Structural rules first (high salience) decide the encounter's shape.
    Rule("boss_floor",   100, lambda f: f["depth"] % 5 == 0,
         lambda f: f.update(kind="boss", enemies=1, tags=f["tags"] | {"elite"})),
    Rule("swarm",         50, lambda f: f["depth"] % 5 != 0 and f["danger"] >= 3,
         lambda f: f.update(kind="swarm", enemies=min(3 + f["depth"], 12))),
    Rule("patrol",        40, lambda f: f["depth"] % 5 != 0 and f["danger"] < 3,
         lambda f: f.update(kind="patrol", enemies=2)),
    # Biome rules layer hazards on top of whatever shape was chosen.
    Rule("lava_hazard",   30, lambda f: f["biome"] == "volcano" and "hazard" not in f,
         lambda f: f.update(hazard="lava")),
    Rule("ice_hazard",    30, lambda f: f["biome"] == "glacier" and "hazard" not in f,
         lambda f: f.update(hazard="ice")),
    # Reward depends on `kind`, which an earlier rule set -> forward chaining.
    Rule("scale_reward",  10, lambda f: "kind" in f and "reward" not in f,
         lambda f: f.update(reward={"boss": "legendary", "swarm": "rare"}.get(f["kind"], "common"))),
    # Lowest-salience default guarantees every region gets loot.
    Rule("default_loot",   1, lambda f: "loot" not in f,
         lambda f: f.update(loot="gold")),
]

def run(rules, facts):
    """Recognize-act cycle with salience-based conflict resolution + forward chaining."""
    fired, order = set(), []
    ranked = sorted(rules, key=lambda r: -r.salience)
    changed = True
    while changed:
        changed = False
        for rule in ranked:
            if rule.name not in fired and rule.when(facts):
                rule.then(facts)                # act -> may enable a lower rule next pass
                fired.add(rule.name)
                order.append(rule.name)
                changed = True
                break                           # restart scan: highest-salience first again
    return facts, order

regions = [
    {"depth": 5,  "danger": 4, "biome": "volcano", "tags": set()},
    {"depth": 3,  "danger": 4, "biome": "glacier", "tags": set()},
    {"depth": 2,  "danger": 1, "biome": "cavern",  "tags": set()},
    {"depth": 10, "danger": 2, "biome": "volcano", "tags": set()},
]

for region in regions:
    facts, order = run(rules, dict(region))
    desc = {k: v for k, v in facts.items() if k not in ("tags",)}
    print(f"depth {facts['depth']:>2} {facts['biome']:<8} -> {desc['kind']:<6} "
          f"enemies={desc['enemies']:<2} reward={desc['reward']:<9} "
          f"hazard={desc.get('hazard','-'):<5} loot={desc['loot']}")
    print(f"   rules fired: {' -> '.join(order)}")

depth  5 volcano  -> boss   enemies=1  reward=legendary hazard=lava  loot=gold
   rules fired: boss_floor -> lava_hazard -> scale_reward -> default_loot
depth  3 glacier  -> swarm  enemies=6  reward=rare      hazard=ice   loot=gold
   rules fired: swarm -> ice_hazard -> scale_reward -> default_loot
depth  2 cavern   -> patrol enemies=2  reward=common    hazard=-     loot=gold
   rules fired: patrol -> scale_reward -> default_loot
depth 10 volcano  -> boss   enemies=1  reward=legendary hazard=lava  loot=gold
   rules fired: boss_floor -> lava_hazard -> scale_reward -> default_loot


Each region drives a different chain of firings, yet the logic lives in seven readable rules.
Note `scale_reward` can only fire *after* a `kind` rule ran — the engine's re-scan handles that
dependency automatically. Want bosses to drop two rewards? Edit one rule; nothing else changes.
That locality is what rule-based generation buys you over a monolithic generator function.

### Optional — let an LLM author rules (gated, no key needed to run)

A common hybrid: keep the deterministic rule *engine*, but have an LLM *propose* new rules or
fill terminal slots. The network call is gated behind an env check so the notebook still runs
top-to-bottom without a key — you see the call shape, not a hard dependency.

In [4]:
# Gate any API/large-download work so the notebook executes with or without credentials.
if os.getenv("ANTHROPIC_API_KEY"):
    from anthropic import Anthropic  # pip install anthropic
    client = Anthropic()
    msg = client.messages.create(
        model="claude-opus-4-8",
        max_tokens=300,
        messages=[{"role": "user", "content":
                   "Propose 3 IF-THEN dungeon decoration rules as JSON "
                   "(condition on a floor cell's neighbors -> feature to place)."}],
    )
    print(msg.content[0].text)
else:
    print("ANTHROPIC_API_KEY not set - skipping live call.")
    print("Pattern: LLM proposes rules as data; your engine still applies them deterministically.")

ANTHROPIC_API_KEY not set - skipping live call.
Pattern: LLM proposes rules as data; your engine still applies them deterministically.


## 6. Gotchas & Pitfalls

- **Non-termination.** If a rule's action keeps its own condition true (or two rules undo each
  other), the recognize–act loop never quiesces. Design rules to *consume* what they match (leave
  the guard) or cap the step count. Both examples above provably reach a fixpoint.
- **Order dependence / conflict-resolution surprises.** When several rules match, *which fires*
  changes the result. Relying on incidental list order is fragile — make priority explicit with
  salience, and write tests that pin the firing order.
- **Emergent interactions don't scale.** Ten rules are readable; a hundred develop combinatorial
  interactions no one fully predicts (the expert-system maintenance trap). Group rules into
  phases/salience bands, and keep each rule's effect local and testable.
- **Sequential vs parallel mismatch.** Applying rules one-at-a-time vs all-at-once against a
  snapshot gives *different* maps. Decide which model you want; a CA/L-system needs the parallel,
  double-buffered version, not in-place mutation mid-pass.
- **Conditions that read mutated state mid-pass.** In-place updates can let a cell "see" a feature
  a neighbor placed *this* pass, biasing results. Use a snapshot (read old, write new) when you
  need pass-consistent behavior.
- **No global guarantees.** Local rules can't ensure "the level is fully connected" or "exactly
  one boss." Add explicit *constraint* rules that detect+repair violations, or post-validate.
- **Silent no-match.** A rule that never fires (typo in the condition, unreachable pattern) fails
  silently. Log firing counts per rule, as Example 2 does, to catch dead rules.
- **Reproducibility.** Any randomness in conflict resolution or weighted rules must be seeded
  (`random.Random(seed)`) for deterministic tests and save-files.

## 7. When to Use vs Alternatives

**Reach for rule-based generation when** the logic is *authorable as discrete if-then design
intent*, you need it *explainable and editable* by hand, and you have no training data. Map
decoration/validation, encounter/loot/quest assembly, post-processing passes, and tutorial-y
systems where a designer must read and tune the behavior.

| Approach | Strength | Weakness vs. rules | Use when |
|---|---|---|---|
| **Rule-based / production system** | Authorable, explainable, no data; rules are local & testable | Emergent interactions at scale; no global guarantees; order-sensitive | Logic is discrete if-then design intent you must edit by hand |
| **Context-free grammar** ([[context-free-grammars]]) | Guaranteed recursive *structure* (text/syntax) | Specialized to symbol rewriting; awkward for spatial/numeric logic | Output is nested text obeying a grammar |
| **L-system** ([[l-systems]]) | Parallel rewriting for fractal/organic growth | Narrow application; all-rules-at-once model | Plants, fractals, recursive geometry |
| **Cellular automata** ([[cellular-automata]]) | Emergent organic shapes from one local rule | Hard to author *specific* outcomes | Caves/erosion via neighbor rules |
| **Wave Function Collapse** ([[wave-function-collapse]]) | Solves hard local tile constraints globally | Heavier; can backtrack/fail; tiles only | Tile maps where neighbors must be compatible |
| **Markov chain** ([[markov-chains]]) | Learns statistics from examples; trivial to train | No explicit control; no structure guarantees | You have examples and want statistical mimicry |
| **Neural / LLM** | Open-ended, semantic, coherent | Needs data/compute; opaque; no hard guarantees | Meaning matters and rules can't capture it |

Rule of thumb: **rules for authored control, grammars/L-systems for recursive structure, CA for
emergent texture, learned models for statistics/meaning.** They compose — a rule engine is a great
*orchestrator* that calls a grammar to fill a slot or an LLM to write a terminal, while keeping
the overall structure deterministic and editable.

## 8. Resources

- **"Production system" — Wikipedia** — the formal recognize–act model, working memory, and
  conflict resolution that all rule engines share: <https://en.wikipedia.org/wiki/Production_system_(computer_science)>
- **"Rule-based system" — Wikipedia** — the broader paradigm and its place vs learned systems:
  <https://en.wikipedia.org/wiki/Rule-based_system>
- **Procedural Content Generation Book (Shaker, Togelius, Nelson)** — free online textbook; the
  search-/grammar-/rule-based chapters frame PCG techniques: <http://pcgbook.com/>
- **PCG Wiki / "Procedural Generation" — Roguelike Dev** — practical map-decoration and
  rule-driven dungeon patterns: <http://www.roguebasin.com/index.php/Procedural_Content_Generation>
- **`experta` docs** — a maintained Python CLIPS-style forward-chaining engine when you outgrow a
  hand-rolled loop: <https://experta.readthedocs.io/>
- **CLIPS** — the classic production-rule language; the reference model for salience and the
  recognize–act cycle: <https://www.clipsrules.net/>